In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view


# ===================================================================== CONFIG
RUNS = {
    0: [
        ("Well_3_1405", "data/0/WELL-00003_20170514010000.csv"),
        ("Well_7_0108", "data/0/WELL-00007_20170801180000.csv"),
        ("Well_5_1708", "data/0/WELL-00005_20170817020000.csv"),
        ("Well_6_2008", "data/0/WELL-00006_20170820010000.csv"),
    ],
    1: [
        ("Well_6_0108", "data/1/WELL-00006_20170801063614.csv"),
        ("Drawn_02", "data/1/Drawn_00002.csv"),
        ("Sim_0112", "data/1/SIMULATED_00112.csv"),
        ("Sim_0044", "data/1/SIMULATED_00044.csv"),
    ],
    2: [
        ("Well_10_1812", "data/2/WELL-00010_20171218200131.csv"),
        ("Well_3_0602", "data/2/WELL-00003_20180206182917.csv"),
        ("Well_12_2003", "data/2/WELL-00012_20170320033022.csv"),
        ("Sim_02", "data/2/SIMULATED_00002.csv"),
    ],
    3: [
        ("Well_14_2810", "data/3/WELL-00014_20171028080000.csv"),
        ("Well_14_1709", "data/3/WELL-00014_20170917150000.csv"),
        ("Well_14_2509", "data/3/WELL-00014_20170925010031.csv"),
        ("Sim_67", "data/3/SIMULATED_00067.csv"),
    ],
    4: [
        ("Well_1_1603", "data/4/WELL-00001_20170316120203.csv"),
        ("Well_7_1705", "data/4/WELL-00007_20170517210012.csv"),
        ("Well_10_1604", "data/4/WELL-00010_20180416080150.csv"),
        ("Well_14_1509", "data/4/WELL-00014_20170915090021.csv"),
    ],
    5: [
        ("Sim_17", "data/5/SIMULATED_00017.csv"),
        ("Well_16_1705", "data/5/WELL-00016_20180517222322.csv"),
        ("Well_16_0504", "data/5/WELL-00016_20180405020345.csv"),
        ("Well_17_1703", "data/5/WELL-00017_20140317151743.csv"),
    ],
    6: [
        ("Sim_2", "data/6/SIMULATED_00002.csv"),
        ("Sim_29", "data/6/SIMULATED_00029.csv"),
        ("Sim_207", "data/6/SIMULATED_00207.csv"),
        ("Sim_111", "data/6/SIMULATED_00111.csv"),
    ],
    7: [
        ("Drawn_10", "data/7/DRAWN_00010.csv"),
        ("Drawn_7", "data/7/DRAWN_00007.csv"),
        ("Drawn_1", "data/7/DRAWN_00001.csv"),
        ("Drawn_3", "data/7/DRAWN_00003.csv"),
    ],
    8: [
        ("Well_21_0905", "data/8/WELL-00021_20170509013517.csv"),
        ("Well_20_1004", "data/8/WELL-00020_20120410192326.csv"),
        ("Sim_00016", "data/8/SIMULATED_00016.csv"),
        ("Sim_00046", "data/8/SIMULATED_00046.csv"),
    ],
}

OUTPUT_ROOT = "../outputs"
SHOW_PLOTS = False

ISO_SCALER_PATH = "../../models/anomaly_scaler.joblib"
ISO_MODEL_PATH = "../../models/anomaly_isolation_forest.joblib"
LSTM_PATH = "../../models/lstm.pth"
LSTM_SCALER_PATH = "../../models/lstm_scaler.joblib"

ISO_FEATURES = ['rel_tot_dp_change', 'rel_ts_dp_change',
                'rel_p_ts_dp_change', 'Relative_Ratio_Dev']
LSTM_FEATURES = ISO_FEATURES + ['anomaly_score']
WINDOW = 60

HOLD = 60        
TOL = 300      
STEADY_MIN = 600 
RRD_STATE_THRESH = {2: None, 3: None}
# =====================================================================

if not SHOW_PLOTS:
    matplotlib.use("Agg")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class PlainLSTM(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=32, num_layers=1, num_classes=4, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :]))


# ---------- load models once ----------
iso_scaler = joblib.load(ISO_SCALER_PATH)
iso_model = joblib.load(ISO_MODEL_PATH)
lstm_scaler = joblib.load(LSTM_SCALER_PATH)
model = PlainLSTM(input_dim=len(LSTM_FEATURES), hidden_dim=32, num_layers=1,
                  num_classes=4, dropout=0.3).to(device)
model.load_state_dict(torch.load(LSTM_PATH, map_location=device))
model.eval()

P_RAW = ['P-PDG', 'P-TPT', 'P-MON-CKP']


# ---------- processing ----------
def load_and_featurize(path):
    df = pd.read_csv(path)
    df = df.drop(columns=['timestamp', 'T-TPT', 'T-JUS-CKP', 'T-JUS-CKGL',
                          'P-JUS-CKGL', 'QGL'], errors='ignore')

    nan_share = {c: float(df[c].isna().mean()) for c in P_RAW}
    frozen_share = {c: float((df[c].diff() == 0).sum() / max(int(df[c].notna().sum()), 1)) for c in P_RAW}

    tmp = df.rename(columns={'P-PDG': 'Pump outlet pressure (Mean)',
                             'P-TPT': 'TS inlet pressure (Mean)',
                             'P-MON-CKP': 'TS outlet pressure (Mean)'})

    if 'class' in tmp.columns:
        raw = tmp['class'].values
        tmp['class_raw'] = raw
        tmp['phase'] = np.select([raw == 0, raw > 100, raw > 0],
                                 ['normal', 'transient', 'steady'], default='unlabeled')
        tmp['class'] = np.where(tmp['class'] > 100, (tmp['class'] - 100) / 2, tmp['class'])
    else:
        tmp['class_raw'] = np.nan
        tmp['phase'] = 'unlabeled'

    p_cols = ['Pump outlet pressure (Mean)', 'TS inlet pressure (Mean)', 'TS outlet pressure (Mean)']
    n_nan = int(tmp[p_cols].isna().sum().sum())
    if n_nan:
        print(f"   warning: {n_nan} NaN pressure values, filled with ffill/bfill")
        tmp[p_cols] = tmp[p_cols].ffill().bfill()

    tmp['Differential pressure (Mean)'] = tmp['TS inlet pressure (Mean)'] - tmp['TS outlet pressure (Mean)']
    tmp['Pump to inlet DP'] = tmp['Pump outlet pressure (Mean)'] - tmp['TS inlet pressure (Mean)']
    tmp['Total system DP'] = tmp['Pump outlet pressure (Mean)'] - tmp['TS outlet pressure (Mean)']

    base = tmp.iloc[:60]
    tmp['rel_tot_dp_change'] = tmp['Total system DP'] / base['Total system DP'].median()
    tmp['rel_ts_dp_change'] = tmp['Differential pressure (Mean)'] / base['Differential pressure (Mean)'].median()
    tmp['rel_p_ts_dp_change'] = tmp['Pump to inlet DP'] / base['Pump to inlet DP'].median()

    tmp['Pressure_Ratio'] = (tmp['TS inlet pressure (Mean)'] - tmp['TS outlet pressure (Mean)']) / \
                            (tmp['Pump outlet pressure (Mean)'] - tmp['TS inlet pressure (Mean)'] + 1e-6)
    base_ratio = tmp['Pressure_Ratio'].iloc[:60].median()
    tmp['Relative_Ratio_Dev'] = (tmp['Pressure_Ratio'] / (base_ratio + 1e-6) - 1).abs()

    tmp.attrs['nan_share'] = nan_share
    tmp.attrs['frozen_share'] = frozen_share
    tmp.attrs['baseline_normal_share'] = float((tmp['phase'].iloc[:60] == 'normal').mean())
    tmp.attrs['base_dp'] = {'tot': float(base['Total system DP'].median()),
                            'ts': float(base['Differential pressure (Mean)'].median()),
                            'p_ts': float(base['Pump to inlet DP'].median())}
    return tmp


def add_anomaly_score(tmp):
    X = iso_scaler.transform(tmp[ISO_FEATURES])
    tmp['anomaly_score'] = iso_model.decision_function(X)
    return tmp


def predict_states(tmp, chunk=4096):
    """Same prediction as a per-sample loop, but batched. First WINDOW-1 samples are set to state 0."""
    n = len(tmp)
    preds = np.zeros(n, dtype=int)
    if n >= WINDOW:
        arr = tmp[LSTM_FEATURES].values.astype(float)
        wins = sliding_window_view(arr, WINDOW, axis=0).transpose(0, 2, 1)   # (n-W+1, W, F)
        out = []
        with torch.no_grad():
            for i in range(0, len(wins), chunk):
                w = wins[i:i + chunk]
                F = w.shape[2]
                ws = lstm_scaler.transform(w.reshape(-1, F)).reshape(w.shape)
                x = torch.tensor(ws, dtype=torch.float32).to(device)
                out.append(model(x).argmax(dim=1).cpu().numpy())
        preds[WINDOW - 1:] = np.concatenate(out)
    tmp['predicted_state'] = preds
    return tmp


# ---------- metrics ----------
def first_sustained(pred, thr, start=WINDOW - 1, hold=HOLD):
    """First start index of `hold` consecutive samples with pred >= thr, anywhere in the run."""
    hit = (pred >= thr).astype(int)
    hit[:start] = 0
    if len(hit) < hold:
        return np.nan
    ok = np.convolve(hit, np.ones(hold, dtype=int), 'valid') == hold
    return int(np.argmax(ok)) if ok.any() else np.nan


def first_sustained_in(pred, mask, thr, hold=HOLD):
    """Same, but all `hold` samples must lie inside `mask`."""
    hit = ((pred >= thr) & mask).astype(int)
    if len(hit) < hold:
        return np.nan
    ok = np.convolve(hit, np.ones(hold, dtype=int), 'valid') == hold
    return int(np.argmax(ok)) if ok.any() else np.nan


def sustained_in(pred, mask, thr, hold=HOLD):
    return not np.isnan(first_sustained_in(pred, mask, thr, hold))


def max_sustained(pred, mask, hold=HOLD):
    """Highest state (0-3) that is held for `hold` consecutive samples inside the mask."""
    for thr in (3, 2, 1):
        if sustained_in(pred, mask, thr, hold):
            return thr
    return 0


def phase_stats(tmp):
    pred, ph = tmp['predicted_state'].values, tmp['phase'].values
    valid = np.arange(len(tmp)) >= WINDOW - 1
    out = {'duration_h': round(len(tmp) / 3600, 2)}
    for p in ['normal', 'transient', 'steady']:
        m = (ph == p) & valid
        out[f'n_{p}'] = int((ph == p).sum())
        for k in range(4):
            out[f'{p}_pred{k}'] = round(float((pred[m] == k).mean()), 4) if m.any() else np.nan
        out[f'{p}_mean_score'] = round(float(tmp.loc[m, 'anomaly_score'].mean()), 4) if m.any() else np.nan
        out[f'{p}_mean_rrd'] = round(float(tmp.loc[m, 'Relative_Ratio_Dev'].mean()), 4) if m.any() else np.nan
    tr, st = np.flatnonzero(ph == 'transient'), np.flatnonzero(ph == 'steady')
    out['t_transient_start'] = int(tr[0]) if len(tr) else np.nan
    out['t_steady_start'] = int(st[0]) if len(st) else np.nan
    for thr in (1, 2, 3):

        f = first_sustained(pred, thr)
        out[f'first_state{thr}'] = f
        out[f'delay{thr}_vs_transient'] = f - out['t_transient_start']
        out[f'delay{thr}_vs_steady'] = f - out['t_steady_start']
    out['peak_rrd'] = round(float(tmp['Relative_Ratio_Dev'].rolling(60, min_periods=1).mean().max()), 4)
    return out


def signal_stats(tmp, masks):
    """What the model's inputs actually did in each phase (rolling-60 mean, to match the window length).
    Use it to tell 'the model missed it' from 'there was nothing to see in the pressures'."""
    rrd = tmp['Relative_Ratio_Dev'].rolling(WINDOW, min_periods=1).mean().values
    dev = (tmp['rel_tot_dp_change'] - 1).abs().rolling(WINDOW, min_periods=1).mean().values
    dev_ts = (tmp['rel_ts_dp_change'] - 1).abs().rolling(WINDOW, min_periods=1).mean().values
    dev_pts = (tmp['rel_p_ts_dp_change'] - 1).abs().rolling(WINDOW, min_periods=1).mean().values
    out = {}
    for pname, m in masks.items():
        for fname, v in (('rrd', rrd), ('dev_tot', dev), ('dev_ts', dev_ts), ('dev_pts', dev_pts)):
            if m.any():
                out[f'sig_{fname}_{pname}_med'] = round(float(np.nanmedian(v[m])), 4)
                out[f'sig_{fname}_{pname}_p95'] = round(float(np.nanpercentile(v[m], 95)), 4)
            else:
                out[f'sig_{fname}_{pname}_med'] = np.nan
                out[f'sig_{fname}_{pname}_p95'] = np.nan
    m = masks['steady']
    for k, thr in RRD_STATE_THRESH.items():
        if thr is not None and m.any():
            out[f'steady_rrd_ge_state{k}_share'] = round(float((rrd[m] >= thr).mean()), 4)
    return out


def criteria(tmp, tol=TOL, hold=HOLD):
    pred, ph = tmp['predicted_state'].values, tmp['phase'].values
    idx = np.arange(len(tmp))
    valid = idx >= WINDOW - 1
    onset = np.flatnonzero(np.isin(ph, ['transient', 'steady']))
    steady_idx = np.flatnonzero(ph == 'steady')
    has_event = len(onset) > 0
    if has_event:
        t0 = int(onset[0])                                   # first transient-or-steady sample
        normal_core = valid & (ph == 'normal') & (idx < t0 - tol)
        event = valid & (idx >= t0 - tol)
    else:                                                    # normal-only run
        t0 = np.nan
        normal_core, event = valid, np.zeros(len(tmp), bool)
    transient = valid & (ph == 'transient')
    steady = valid & (ph == 'steady')

    out = {'has_event': has_event, 't_onset': t0,
           'normal_core_n': int(normal_core.sum()), 'steady_n_valid': int(steady.sum())}

    # 1. normal behaviour: false alarm = sustained state >= 2 in the normal core
    f_fa = first_sustained_in(pred, normal_core, 2, hold)
    out['fa_state2_normal'] = not np.isnan(f_fa)
    out['fa_first_state2_idx'] = f_fa
    out['fa_lead_vs_onset'] = f_fa - t0                      # negative = seconds before the label (event runs)
    out['sustained_state1_normal'] = sustained_in(pred, normal_core, 1, hold)
    for k in range(4):
        out[f'normal_core_pred{k}'] = round(float((pred[normal_core] == k).mean()), 4) if normal_core.any() else np.nan

    # 2. response in the event window (t0 - tol ... end of run); delay is measured inside that window
    out['eligible'] = bool((ph[:WINDOW] == 'normal').all())  # screening rule: first 60 samples labeled normal
    for thr in (1, 2, 3):
        f = first_sustained_in(pred, event, thr, hold)
        out[f'hit_state{thr}_event'] = not np.isnan(f)
        out[f'delay{thr}_event'] = f - t0                    # negative = early (>= -tol by construction)
        # delay == -tol means "already at this level when the window opened": the true onset is earlier
        out[f'delay{thr}_censored'] = bool(not np.isnan(f) and has_event and f - t0 == -tol)

    # 3. steady period: under-response and fall-back
    out['steady_max_sustained'] = max_sustained(pred, steady, hold) if steady.any() else np.nan
    out['event_max_sustained'] = max_sustained(pred, event, hold) if has_event else np.nan
    long_steady = bool(steady.sum() >= STEADY_MIN)
    out['long_steady'] = long_steady
    out['under_response'] = bool(long_steady and out['steady_max_sustained'] <= 1)
    out['fell_back'] = bool(out['under_response'] and out['event_max_sustained'] >= 2)

    # one outcome per run
    if not has_event:
        oc = 'false_alarm' if out['fa_state2_normal'] else \
            'state1_only_normal' if out['sustained_state1_normal'] else 'clean'
    elif not out['hit_state1_event']:
        oc = 'miss'
    elif not out['hit_state2_event']:
        oc = 'state1_only'
    else:
        f2 = first_sustained_in(pred, event, 2, hold)
        s0 = steady_idx[0] if len(steady_idx) else np.inf
        oc = 'responded_before_steady' if f2 < s0 else 'responded_in_steady'
    out['outcome'] = oc

    # signal diagnostics
    out.update(signal_stats(tmp, {'normal': normal_core, 'transient': transient, 'steady': steady}))
    return out


def build_tables(summary_df):
    """The numbers for the Results section."""
    df = summary_df[summary_df['error'].isna()].copy() if 'error' in summary_df.columns else summary_df.copy()
    df['has_event'] = df['has_event'].astype(bool)
    tables = {}

    # 1. normal-only runs
    no = df[~df['has_event']]
    cols = ['name', 'source', 'n_samples', 'fa_state2_normal', 'fa_first_state2_idx', 'sustained_state1_normal',
            'normal_pred0', 'normal_pred1', 'normal_pred2', 'normal_pred3', 'outcome']
    t = no[cols].copy()
    if len(no):
        w = (no['n_normal'] - (WINDOW - 1)).clip(lower=0)
        pooled = {f'normal_pred{k}': round(float((no[f'normal_pred{k}'] * w).sum() / w.sum()), 4) for k in range(4)}
        pooled.update({'name': 'POOLED (sample-weighted; fa column = count)',
                       'n_samples': int(no['n_samples'].sum()),
                       'fa_state2_normal': int(no['fa_state2_normal'].astype(bool).sum())})
        t = pd.concat([t, pd.DataFrame([pooled])], ignore_index=True)
    tables['normal_only'] = t

    # 2. event runs that pass the screening rule (first 60 samples normal), per class / overall / per source
    ev_all = df[df['has_event']]
    elig = ev_all['eligible'].astype(bool)
    ev = ev_all[elig]
    tables['excluded_not_normal_start'] = ev_all[~elig][
        ['class_folder', 'name', 'source', 'hit_state1_event', 'hit_state2_event', 'hit_state3_event', 'outcome']]

    def summarize(g, label):
        r = {'group': label, 'n_runs': len(g)}
        for k in (1, 2, 3):
            r[f'resp_ge{k}'] = f"{int(g[f'hit_state{k}_event'].astype(bool).sum())}/{len(g)}"
            d = g[f'delay{k}_event'].dropna()
            r[f'median_delay_ge{k}_s'] = float(d.median()) if len(d) else np.nan
        r['n_ge1_already_at_window_start'] = int(g['delay1_censored'].astype(bool).sum())
        r['pre_onset_alarm'] = int(g['fa_state2_normal'].astype(bool).sum())
        r['under_response'] = int(g['under_response'].astype(bool).sum())
        r['fell_back'] = int(g['fell_back'].astype(bool).sum())
        return r

    rows = [summarize(g, f'class {c}') for c, g in ev.groupby('class_folder')]
    rows.append(summarize(ev, 'ALL EVENT RUNS'))
    rows += [summarize(g, f'source: {s}') for s, g in ev.groupby('source')]
    tables['events'] = pd.DataFrame(rows)

    # 3. outcome counts
    tables['outcomes_events'] = pd.crosstab(ev['class_folder'], ev['outcome'])
    tables['outcomes_normal_only'] = pd.crosstab(no['source'], no['outcome'])
    return tables


# ---------- plotting ----------
def make_plot(df, col, title, ylabel, out_path, smooth_window=100):
    fig, ax = plt.subplots(figsize=(16, 6))
    x = df['elapsed_seconds']
    if smooth_window > 1:
        ax.plot(x, df[col], color='skyblue', alpha=0.4, label=col)
        ax.plot(x, df[col].rolling(window=smooth_window, min_periods=1).mean(),
                color='darkblue', linewidth=2, label='Smoothed Trend')
    else:
        ax.plot(x, df[col], color='darkblue', linewidth=2, label=col)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Time (elapsed seconds)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


def save_all_plots(tmp, name, cls):
    out_dir = os.path.join(OUTPUT_ROOT, str(cls))
    os.makedirs(out_dir, exist_ok=True)
    make_plot(tmp, 'Relative_Ratio_Dev', f'{name} - Relative Ratio Deviation Over Time',
              'Deviation Magnitude', os.path.join(out_dir, f'{name}_relative_ratio_deviation.png'))
    make_plot(tmp, 'anomaly_score', f'{name} - Anomaly Score Over Time',
              'Anomaly Magnitude', os.path.join(out_dir, f'{name}_anomaly_score.png'))
    make_plot(tmp, 'predicted_state', f'{name} - predicted label',
              'label', os.path.join(out_dir, f'{name}_predicted_label.png'), smooth_window=1)
    make_plot(tmp, 'class', f'{name} - actual label',
              'label', os.path.join(out_dir, f'{name}_actual_label.png'), smooth_window=1)


# ---------- main loop ----------
if __name__ == "__main__":
    summary = []
    for cls, runs in RUNS.items():
        for name, path in runs:
            print(f"[class {cls}] {name}")
            try:
                tmp = load_and_featurize(path)
                tmp = add_anomaly_score(tmp)
                tmp = predict_states(tmp)
                tmp['elapsed_seconds'] = tmp.index.astype(int)
                

                out_dir = os.path.join(OUTPUT_ROOT, str(cls))
                os.makedirs(out_dir, exist_ok=True)
                tmp[['elapsed_seconds', 'class_raw', 'phase', 'predicted_state',
                     'anomaly_score', 'Relative_Ratio_Dev']].to_csv(
                    os.path.join(out_dir, f'{name}_series.csv.gz'), index=False)

                shares = tmp['predicted_state'].value_counts(normalize=True)
                summary.append({
                    'class_folder': cls, 'name': name, 'n_samples': len(tmp),
                    'source': 'sim' if name.startswith('Sim') else 'drawn' if name.startswith('Drawn') else 'real',
                    **{f'pred_{k}_share': round(float(shares.get(k, 0)), 4) for k in range(4)},
                    'mean_anomaly_score': round(float(tmp['anomaly_score'].mean()), 4),
                    'min_anomaly_score': round(float(tmp['anomaly_score'].min()), 4),
                    # diagnostics
                    'baseline_normal_share': tmp.attrs['baseline_normal_share'],
                    'baseline_dp_nonpositive': any(v <= 0 for v in tmp.attrs['base_dp'].values()),
                    **{f'base_dp_{k}': v for k, v in tmp.attrs['base_dp'].items()},
                    **{f'nan_{k}': round(v, 4) for k, v in tmp.attrs['nan_share'].items()},
                    **{f'frozen_{k}': round(v, 4) for k, v in tmp.attrs['frozen_share'].items()},
                    # per-phase statistics, timing, criteria, outcome, signal diagnostics
                    **phase_stats(tmp),
                    **criteria(tmp),
                })
            except Exception as e:
                print(f"   FAILED: {e}")
                summary.append({'class_folder': cls, 'name': name, 'error': str(e)})

    summary_df = pd.DataFrame(summary)
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    summary_df.to_csv(os.path.join(OUTPUT_ROOT, 'summary.csv'), index=False)

    pd.set_option('display.width', 250, 'display.max_columns', 100)
    if 'outcome' in summary_df.columns:
        print(summary_df[['class_folder', 'name', 'source', 'outcome', 'fa_state2_normal', 'fa_lead_vs_onset',
                          'hit_state1_event', 'hit_state2_event', 'hit_state3_event',
                          'delay1_event', 'delay2_event', 'delay3_event',
                          'steady_max_sustained', 'under_response', 'fell_back']])
        tables = build_tables(summary_df)
        for key, tbl in tables.items():
            tbl.to_csv(os.path.join(OUTPUT_ROOT, f'table_{key}.csv'), index=(key.startswith('outcomes')))
            print(f"\n=== {key} ===")
            print(tbl.to_string())
    else:
        print(summary_df)

[class 0] Well_3_1405
[class 0] Well_7_0108
[class 0] Well_5_1708
[class 0] Well_6_2008
[class 1] Well_6_0108
[class 1] Drawn_02
[class 1] Sim_0112
[class 1] Sim_0044
[class 2] Well_10_1812
[class 2] Well_3_0602
[class 2] Well_12_2003
[class 2] Sim_02
[class 3] Well_14_2810
[class 3] Well_14_1709
[class 3] Well_14_2509
[class 3] Sim_67
[class 4] Well_1_1603
[class 4] Well_7_1705
[class 4] Well_10_1604
[class 4] Well_14_1509
[class 5] Sim_17
[class 5] Well_16_1705
[class 5] Well_16_0504
[class 5] Well_17_1703
[class 6] Sim_2
[class 6] Sim_29
[class 6] Sim_207
[class 6] Sim_111
[class 7] Drawn_10
[class 7] Drawn_7
[class 7] Drawn_1
[class 7] Drawn_3
[class 8] Well_21_0905
[class 8] Well_20_1004
[class 8] Sim_00016
[class 8] Sim_00046
    class_folder          name source                  outcome  fa_state2_normal  fa_lead_vs_onset  hit_state1_event  hit_state2_event  hit_state3_event  delay1_event  delay2_event  delay3_event  steady_max_sustained  under_response  fell_back
0             

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "../outputs")
FIG_DIR = os.path.join(OUTPUT_ROOT, "figures")

FIGURE_GROUPS = {
    "fig_3w_class0": [(0, "Well_3_1405"), (0, "Well_7_0108"), (0, "Well_5_1708"), (0, "Well_6_2008")],
    "fig_3w_class1": [(1, "Well_6_0108"), (1, "Drawn_02"), (1, "Sim_0112"), (1, "Sim_0044")],
    "fig_3w_class2": [(2, "Well_10_1812"), (2, "Well_3_0602"), (2, "Well_12_2003"), (2, "Sim_02")],
    "fig_3w_class3": [(3, "Well_14_2810"), (3, "Well_14_1709"), (3, "Well_14_2509"), (3, "Sim_67")],
    "fig_3w_class4": [(4, "Well_1_1603"), (4, "Well_7_1705"), (4, "Well_10_1604"), (4, "Well_14_1509")],
    "fig_3w_class5": [(5, "Sim_17"), (5, "Well_16_1705"), (5, "Well_16_0504"), (5, "Well_17_1703")],
    "fig_3w_class6": [(6, "Sim_2"), (6, "Sim_29"), (6, "Sim_207"), (6, "Sim_111")],
    "fig_3w_class7": [(7, "Drawn_10"), (7, "Drawn_7"), (7, "Drawn_1"), (7, "Drawn_3")],
    "fig_3w_class8": [(8, "Well_21_0905"), (8, "Well_20_1004"), (8, "Sim_00016"), (8, "Sim_00046")],

}
SHOW_SCORE = True        
WINDOW = 60              
HOLD = 60               
TOL = 300               
MARK_LEVELS = ()         
FIG_WIDTH_IN = 6.5      
COMMON_TIME_AXIS = False 
# =====================================================================

INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e3e2de"
LINE = "#2a78d6"                                      
PHASE_FILL = {"transient": ("#eda100", 0.26), "steady": ("#e8522f", 0.34)}   
NOLABEL_FILL = ("#cfcec9", 0.7)                       
PHASE_TEXT = {"normal": "Normal", "transient": "Transient", "steady": "Steady"}

plt.rcParams.update({"font.size": 8, "axes.edgecolor": INK_2, "axes.labelcolor": INK_2,
                     "xtick.color": INK_2, "ytick.color": INK_2, "pdf.fonttype": 42})


def spans(phase):
    """Consecutive stretches of equal phase -> list of (phase, first_index, end_index_exclusive)."""
    p = np.asarray(phase)
    change = np.flatnonzero(p[1:] != p[:-1]) + 1
    starts, ends = np.r_[0, change], np.r_[change, len(p)]
    return [(p[s], int(s), int(e)) for s, e in zip(starts, ends)]


def first_sustained(pred, thr, start=WINDOW - 1, hold=HOLD):
    hit = (pred >= thr).astype(int)
    hit[:start] = 0
    if len(hit) < hold:
        return None
    ok = np.convolve(hit, np.ones(hold, dtype=int), "valid") == hold
    return int(np.argmax(ok)) if ok.any() else None


def style_axis(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(axis="y", color=GRID, lw=0.6, zorder=0.5)
    ax.set_axisbelow(True)
    ax.margins(x=0)


def draw_run(ax, ax_score, df, title, xlim):
    x = df["elapsed_seconds"].to_numpy()
    ph = df["phase"].astype(str).to_numpy()
    pred = df["predicted_state"].to_numpy()
    valid = np.arange(len(df)) >= WINDOW - 1
    sp = spans(ph)
    axes = [a for a in (ax, ax_score) if a is not None]

    # phase shading on every panel, labelled once above the top panel
    for a in axes:
        for p, s, e in sp:
            if p in PHASE_FILL:
                a.axvspan(x[s], x[e - 1] + 1, facecolor=PHASE_FILL[p][0], alpha=PHASE_FILL[p][1], lw=0, zorder=0)
    span_total = xlim[1] - xlim[0]
    for p, s, e in sp:
        if p in PHASE_TEXT and (x[e - 1] + 1 - x[s]) > 0.08 * span_total:
            ax.text((x[s] + x[e - 1] + 1) / 2, 1.02, PHASE_TEXT[p], transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", fontsize=8, color=INK_2)

    # predicted state (from the first full window on)
    ax.step(x[valid], pred[valid], where="post", color=LINE, lw=1.6, zorder=2)
    ax.set_ylim(-0.3, 3.3)
    ax.set_yticks([0, 1, 2, 3])
    ax.set_ylabel("Predicted state")


    for k in MARK_LEVELS:
        i = first_sustained(pred, k)
        if i is not None:
            ax.plot(x[i], pred[i], marker="o", ms=5.5, mfc=LINE, mec="white", mew=1.0, zorder=3, ls="none")
            # label below-right of the marker: clear of the rising step and of the left axis
            ax.annotate(f"≥{k}", (x[i], pred[i]), xytext=(5, -4), textcoords="offset points",
                        ha="left", va="top", fontsize=8, color=INK)

    ax.set_title(title, loc="left", fontsize=9, color=INK, pad=16)
    style_axis(ax)

    if ax_score is not None:
        roll = df["anomaly_score"].rolling(WINDOW, min_periods=1).mean().to_numpy()
        ax_score.plot(x[valid], roll[valid], color=INK_2, lw=1.4, zorder=2)
        # decision boundary of the Isolation Forest: scores below 0 are classified as outliers
        ax_score.axhline(0, color=INK_2, lw=1.0, ls=(0, (1, 2.5)), alpha=0.7, zorder=1.8)
        lo, hi = min(roll[valid].min(), 0.0), max(roll[valid].max(), 0.0)
        pad = 0.08 * (hi - lo)
        ax_score.set_ylim(lo - pad, hi + pad)
        ax_score.text(1.008, 0, "threshold", transform=ax_score.get_yaxis_transform(),
                      ha="left", va="center", fontsize=7, color=INK_2, clip_on=False)
        ax_score.set_ylabel("Anomaly\nscore", fontsize=8)
        style_axis(ax_score)
        for a in (ax, ax_score):
            a.yaxis.set_label_coords(-0.075, 0.5)
        ax.tick_params(labelbottom=False)
    axes[-1].set_xlabel("Time since start of run (s)")
    for a in axes:                      
        a.set_xlim(*xlim)


def make_figure(runs, stem):
    frames = []
    for cls, name in runs:
        path = os.path.join(OUTPUT_ROOT, str(cls), f"{name}_series.csv.gz")
        if not os.path.exists(path):
            print(f"missing: {path}")
            continue
        frames.append((cls, name, pd.read_csv(path)))
    if not frames:
        return
    n = len(frames)
    run_end = [int(df["elapsed_seconds"].max()) + 1 for _, _, df in frames]
    if COMMON_TIME_AXIS:
        run_end = [max(run_end)] * n
    fig = plt.figure(figsize=(FIG_WIDTH_IN, (2.9 if SHOW_SCORE else 2.1) * n + 0.2))
    outer = fig.add_gridspec(n, 1, hspace=0.62)
    for i, (cls, name, df) in enumerate(frames):
        if SHOW_SCORE:
            inner = outer[i].subgridspec(2, 1, height_ratios=[3, 1.4], hspace=0.08)
            ax = fig.add_subplot(inner[0])
            ax_score = fig.add_subplot(inner[1], sharex=ax)
        else:
            ax, ax_score = fig.add_subplot(outer[i]), None
        draw_run(ax, ax_score, df, f"{name} (class {cls})", (0, run_end[i]))
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, stem + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, stem + ".png"), dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {stem}")


if __name__ == "__main__":
    for stem, runs in FIGURE_GROUPS.items():
        make_figure(runs, stem)

saved fig_3w_class0
saved fig_3w_class1
saved fig_3w_class2
saved fig_3w_class3
saved fig_3w_class4
saved fig_3w_class5
saved fig_3w_class6
saved fig_3w_class7
saved fig_3w_class8
